# Study guide: run prediction on a new document

This notebook uses the saved vectorizer and classifiers from `train_models.ipynb` to classify a sample file. It also contains an optional LLM explanation example.

Run it from the `notebooks/` directory with the project `.venv313` environment. The first two code sections are local and do not need an API key. The LLM section needs a `TOGETHER_API_KEY` environment variable.


### Cell 1: read and clean a document

This cell defines a format-aware `read_file` function for TXT, PDF, and DOCX files. It reads the selected sample, converts it to plain text, applies the same cleaner used during training, and previews the first 100 cleaned characters.

Change `doc_path` to try `../data/sample.txt` or `../data/sample_doc.docx`.


In [7]:
import re
import os
import fitz
from docx import Document


def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text

def read_file(doc_path):
    ext = os.path.splitext(doc_path)[1].lower()

    if ext == ".txt":
        with open(doc_path, "r", encoding="utf-8") as f:
            return f.read()
        

    elif ext == ".pdf":
        text = ""
        with fitz.open(doc_path) as doc:
            for page in doc:
                text += page.get_text()

        return text
    
    elif ext == ".docx":
        doc = Document(doc_path)
        return "\n".join([para.text for para in doc.paragraphs])
    

    else:
        raise ValueError(f"Unsupported file type: {ext}")
    

doc_path = "../data/sample_pdf.pdf"

raw_text = read_file(doc_path)

new_text_clean = clean_text(raw_text)
new_text_clean[:100]

'over the past year multinational corporations have undergone major \nstrategic transformations to ada'

### Cell 2: load artifacts and predict

This cell loads the fitted TF-IDF vectorizer and all three saved classifiers. It transforms the new document, asks each model for a label and probability, prints the chosen model's result, and writes all results to `../reports/classification_results.csv`.

The vectorizer must be the one created during training. Do not fit a new vectorizer here.


In [8]:
import joblib
import os
from IPython.display import FileLink
vectorizer = joblib.load("tfidf_vectorizer.joblib")
vec_new_text = vectorizer.transform([new_text_clean])

models ={
    "Logistic Regression": joblib.load('../models/logistic_regression.pkl'),
    "SVM": joblib.load('../models/svm.pkl'),
    "Random Forest": joblib.load('../models/random_forest.pkl')
}
results = {}

for name,model in models.items():
    pred = model.predict(vec_new_text)[0]
    prob = max(model.predict_proba(vec_new_text)[0])
    results[name] = (pred, prob)


chosen_model = "SVM"
predicted_label = results[chosen_model][0]
probability = results[chosen_model][1]

print(f"📄 Document: {doc_path}")
print(f"🧠 Predicted Label ({chosen_model}): {predicted_label} (Confidence: {probability:.2f})")

output_path = '../reports/classification_results.csv'
os.makedirs('../reports', exist_ok=True)

with open(output_path, 'w') as f:
    f.write("Model,Predicted Label,Confidence\n")
    for name, (label, prob) in results.items():
        f.write(f"{name},{label},{prob:.4f}\n")

FileLink(output_path)



📄 Document: ../data/sample_pdf.pdf
🧠 Predicted Label (SVM): business (Confidence: 0.88)


c:\Users\acer\Desktop\Document Analyzer\reports\classification_results.csv

### Cell 3: blank workspace cell

This cell is intentionally empty. Use it to inspect the `results` dictionary or compare the three model probabilities.


### Cell 4: optional LLM explanation

This cell sends the first 2,000 characters and the selected category to Together AI through LangChain. Set your key before running it:

~~~bash
export TOGETHER_API_KEY="your-key"
~~~

The LLM explains the title and category; it does not change the classifier prediction.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate


# 🔑 Get API key (only first time)
api_key = os.environ.get("TOGETHER_API_KEY")
if not api_key:
    raise RuntimeError("Set TOGETHER_API_KEY before running this cell")

# Setup LangChain LLM (using LLaMA 3 model on Together)
llm = ChatOpenAI(
    model="meta-llama/Llama-3-8b-chat-hf",  
    temperature=0.7,
    api_key=api_key,
    base_url="https://api.together.xyz/v1"
)
# Create prompt
prompt = PromptTemplate.from_template("""
You are a document analysis assistant.

Document:
{text}

Predicted Category: {category}

1. Generate a suitable, engaging title for this document.
2. Explain in 1-2 sentences **why this is a good title**.
3. Explain in 1-2 sentences **why this document fits the category "{category}"**.
""")

# Generate response
response = llm.invoke(prompt.format(text=new_text_clean[:2000], category=predicted_label))  # limit if very long

print("🎯 LangChain Output:\n")
print(response.content)


🎯 LangChain Output:

**Title:** "Adapting to the New Normal: The Evolution of Business Strategy in a Post-Pandemic World"

**Why this is a good title:** This title effectively captures the essence of the document, highlighting the theme of adaptation and transformation in the business world. It's also engaging and attention-grabbing, making it suitable for a business audience.

**Why this document fits the category "business":** This document is categorized as business because it discusses the strategic transformations and trends in the global business landscape, including topics such as supply chain disruptions, digital transformation, and sustainable practices. The language used is technical and industry-specific, indicating that the document is intended for a professional or academic audience within the business sector.


## Dataset source

The BBC News dataset used by the training notebook is available on [Kaggle](https://www.kaggle.com/datasets/moazeldsokyx/bbc-news). This is a reference link, not executable Python.
